# Property agent test

Runs the property extraction agents in `synth_extract/agents/extraction/` on one paper:

- `p2`: `PaperPropertyExtractor`, per paper, values linked to samples (batches of target samples, usually one request).
- `p3`: `SamplePropertyExtractor`, one request per sample, the other samples as context.

Set the paper, property, and prompt in **Inputs**, then run all cells. The prompts and schemas come from the agent package
(`property_config.py` and `prompts/property/`); the agent cell reloads them, so edits there apply without restarting the kernel.

In [1]:
import sys
sys.dont_write_bytecode = True
import asyncio
import importlib
import json
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

# Find the repository from this folder or any parent.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/polymer_samples.db").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

BASE_URL = "http://localhost:8000/v1"
MODEL = "qwen3.8-27b"
THINKING = True          # Set False for thinking off.
MAX_TOKENS = 20_000      # Completion tokens per request.
TIMEOUT = 1200           # Seconds per request.
SHOW_REASONING = False   # Print the model's reasoning trace for each request.

## Inputs

In [2]:
PAPER_UID = "ID000752848"
PROPERTY = "melting_temperature"  # glass_transition_temperature, melting_temperature, thermal_decomposition_temperature,
                                           # number_average_molecular_weight, weight_average_molecular_weight, dispersity
PROMPT = "p2"             # "p2" (per paper) or "p3" (per sample, with context)
SAMPLE_IDS = None         # p3 only: a list of sample_ids to run, or None for every sample
BATCH_SIZE = 15           # p2 only: target samples per request
MAX_CONCURRENCY = 4       # parallel requests
SHOW_PROMPT = True        # print the system prompt and the first user prompt (paper text shortened)

## Load the paper

Samples come from `polymer_samples.db` (read-only); the full text from `data/fulltext/<source>/<uid>/<uid>.md`.

In [3]:
def read_db(query, params=()):
    connection = sqlite3.connect(f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True)
    try:
        return pd.read_sql_query(query, connection, params=params)
    finally:
        connection.close()


def load_paper(paper_uid):
    from synth_extract.agents.extraction import PaperContext, SampleRef
    rows = read_db(
        "SELECT s.sample_id, s.name, s.identifier, s.aliases, s.description, p.canonical_source AS source, p.title "
        "FROM samples s JOIN papers p USING (paper_uid) WHERE s.paper_uid = ? ORDER BY s.sample_id",
        (paper_uid,),
    )
    if rows.empty:
        raise ValueError(f"No samples for {paper_uid}.")
    md_path = ROOT / "data/fulltext" / rows.source.iloc[0] / paper_uid / f"{paper_uid}.md"
    markdown = md_path.read_text(encoding="utf-8")
    samples = [SampleRef(sample_id=int(r.sample_id), name=r.name, identifier=r.identifier,
                         aliases=r.aliases, description=r.description) for r in rows.itertuples()]
    paper = PaperContext(paper_uid=paper_uid, samples=samples, markdown_full_text=markdown)
    return paper, rows.title.iloc[0] or "[no title in database]", rows


def property_presence(paper_uid):
    rows = read_db("SELECT status, properties_json FROM property_classification WHERE paper_uid = ?", (paper_uid,))
    if rows.empty or rows.status.iloc[0] != "success":
        return []
    return [name.strip().lower() for name in json.loads(rows.properties_json.iloc[0])]


paper, title, sample_rows = load_paper(PAPER_UID)
print(f"{PAPER_UID}: {title}")
print(f"{len(paper.samples)} samples; full text {len(paper.markdown_full_text):,} characters")
present = property_presence(PAPER_UID)
print("Property presence:", ", ".join(present) or "none")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(sample_rows[["sample_id", "name", "identifier", "aliases", "description"]])

ID000752848: Trans crystallization behavior and strong reinforcement effect of cellulose nanocrystals on reinforced poly(butylene succinate) nanocomposites.
5 samples; full text 31,358 characters
Property presence: number-average molecular weight, weight-average molecular weight, dispersity, degree of polymerization, glass transition temperature, melting temperature, thermal decomposition temperature, young's modulus, tensile strength, elongation at break, melt viscosity


,sample_id,name,identifier,aliases,description
0,1764764,Homo PBS,Homo PBS,"[""Homo-PBS"",""PBS""]","Poly(butylene succinate) synthesized via in situ two-step melt-condensation polymerization of succinic acid and 1,4-butanediol without cellulose nanocrystals. The molar ratio of [COOH]/[OH] was fixed at 1.2. Titanium(IV) butoxide (150 ppm relative to total monomer weight) was used as a catalyst. The mixture was heated to 150 °C (2–3 °C min⁻¹) under N₂, then to 200 °C (2–3 °C min⁻¹), and finally to 240 °C under vacuum (<100 mTorr) for a polycondensation time of 2 h 20 min. Stirring was maintained at 150 rpm and decreased to 30 rpm as polymerization progressed. The product was quenched in a water bath and dried in a vacuum oven at 80 °C for 24 h."
1,1764765,PCN01,PCN01,"[""0.1 wt% CNC-loaded PBS nanocomposite""]","Poly(butylene succinate) nanocomposite containing 0.1 wt% spray-dried cellulose nanocrystals (CNCs). Prepared by dispersing CNCs in 1,4-butanediol via ultrasonication for 5 min, followed by in situ two-step melt-condensation polymerization with succinic acid ([COOH]/[OH] molar ratio = 1.2) and titanium(IV) butoxide catalyst (150 ppm). The reaction was heated to 150 °C, then 200 °C, and finally 240 °C under vacuum (<100 mTorr) for a polycondensation time of 2 h 15 min. The product was quenched in a water bath and dried in a vacuum oven at 80 °C for 24 h."
2,1764766,PCN03,PCN03,"[""0.3 wt% CNC-loaded PBS nanocomposite""]","Poly(butylene succinate) nanocomposite containing 0.3 wt% spray-dried cellulose nanocrystals (CNCs). Prepared by dispersing CNCs in 1,4-butanediol via ultrasonication for 5 min, followed by in situ two-step melt-condensation polymerization with succinic acid ([COOH]/[OH] molar ratio = 1.2) and titanium(IV) butoxide catalyst (150 ppm). The reaction was heated to 150 °C, then 200 °C, and finally 240 °C under vacuum (<100 mTorr) for a polycondensation time of 3 h 30 min. The product was quenched in a water bath and dried in a vacuum oven at 80 °C for 24 h."
3,1764767,PCN05,PCN05,"[""0.5 wt% CNC-loaded PBS nanocomposite""]","Poly(butylene succinate) nanocomposite containing 0.5 wt% spray-dried cellulose nanocrystals (CNCs). Prepared by dispersing CNCs in 1,4-butanediol via ultrasonication for 5 min, followed by in situ two-step melt-condensation polymerization with succinic acid ([COOH]/[OH] molar ratio = 1.2) and titanium(IV) butoxide catalyst (150 ppm). The reaction was heated to 150 °C, then 200 °C, and finally 240 °C under vacuum (<100 mTorr) for a polycondensation time of 6 h 35 min. The product was quenched in a water bath and dried in a vacuum oven at 80 °C for 24 h."
4,1764768,PCN10,PCN10,"[""1.0 wt% CNC-loaded PBS nanocomposite""]","Poly(butylene succinate) nanocomposite containing 1.0 wt% spray-dried cellulose nanocrystals (CNCs). Prepared by dispersing CNCs in 1,4-butanediol via ultrasonication for 5 min, followed by in situ two-step melt-condensation polymerization with succinic acid ([COOH]/[OH] molar ratio = 1.2) and titanium(IV) butoxide catalyst (150 ppm). The reaction was heated to 150 °C, then 200 °C, and finally 240 °C under vacuum (<100 mTorr) for a polycondensation time of 16 h 30 min. The product was quenched in a water bath and dried in a vacuum oven at 80 °C for 24 h."


## Build the agent and the requests

In [4]:
# Reload the agent package so edits to the config, templates, or code apply without a kernel restart.
import synth_extract.agents.extraction.property_config as _config
import synth_extract.agents.extraction.property_prompts as _prompts
import synth_extract.agents.extraction.property_schemas as _schemas
import synth_extract.agents.extraction.property_extractor as _extractor
for _module in (_config, _prompts, _schemas, _extractor):
    importlib.reload(_module)
from synth_extract.agents.llm import LLMBackend, StructuredFailure

if PROPERTY not in _config.PROPERTY_CONFIG:
    raise ValueError(f"PROPERTY must be one of {list(_config.PROPERTY_CONFIG)}")
if _config.PROPERTY_CONFIG[PROPERTY]["presence_label"] not in present:
    print(f"Note: {PROPERTY!r} is not flagged by property presence for this paper.")

backend = LLMBackend(
    model=MODEL, base_url=BASE_URL, api_key="none", temperature=0.0,
    max_tokens=MAX_TOKENS, timeout=TIMEOUT,
    extra_body={"chat_template_kwargs": {"enable_thinking": THINKING}},
)
paper = _schemas.PaperContext.model_validate(paper.model_dump())  # rebuild with the reloaded classes

if PROMPT == "p2":
    extractor = _extractor.PaperPropertyExtractor(backend, PROPERTY)
    requests = extractor.build_requests(paper, BATCH_SIZE)
    request_label = lambda r: f"targets {r.target_sample_ids}"
elif PROMPT == "p3":
    extractor = _extractor.SamplePropertyExtractor(backend, PROPERTY)
    requests = extractor.build_requests(paper, SAMPLE_IDS)
    request_label = lambda r: f"sample {r.target.sample_id}"
else:
    raise ValueError('PROMPT must be "p2" or "p3"')

print(f"Agent: {type(extractor).__name__} ({extractor.mode}), property {PROPERTY}")
print(f"Prompt hash: {extractor.prompt_hash()}")
print(f"{len(requests)} request(s):", "; ".join(request_label(r) for r in requests))

if SHOW_PROMPT:
    body = json.loads(extractor.render_request(requests[0]))
    system, user = body["messages"][0]["content"], body["messages"][1]["content"]
    text = paper.markdown_full_text
    user = user.replace(text, f"[... paper text, {len(text):,} characters ...]")
    print("\n" + "=" * 30 + " SYSTEM PROMPT " + "=" * 30 + "\n" + system)
    print("=" * 30 + f" USER PROMPT ({request_label(requests[0])}) " + "=" * 30 + "\n" + user)

Agent: PaperPropertyExtractor (per_paper), property melting_temperature
Prompt hash: 7167b434a6f18e5a8ca9fc81d5d07ddabd2304907adb72e27160037b329a5001
1 request(s): targets [1764764, 1764765, 1764766, 1764767, 1764768]

============================== SYSTEM PROMPT ==============================
# Task

Extract every **melting temperature (Tm)** value reported in the supplied paper and link each value to a sample from the supplied sample list.

The sample list contains every sample recorded for this paper. Use it **only to identify samples**. All values and measurement details must come from the paper, including its text, tables, captions, footnotes, and supplementary text.

Return exactly:

`{"properties": [...]}`

If the paper reports no Tm for the target samples, return `{"properties": []}`.

## 1. Which values to report

Report every Tm value the paper gives for a target sample, and every Tm value that matches no sample in the list. Skip values that clearly belong to a listed sample 

## Call the agent

The requests are sent concurrently (at most `MAX_CONCURRENCY` at a time); `collect` then applies the agent's clean-up and flags, as `aextract_paper` does, and `to_properties()` maps the result to the stored `Property` records.

In [5]:
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)


async def run_one(request):
    async with semaphore:
        return await extractor.aextract(request)

outcomes = await asyncio.gather(*(run_one(r) for r in requests))
result = extractor.collect(paper, requests, outcomes)
properties = result.to_properties()   # the stored records: one per value of a sample

for request, outcome in zip(requests, outcomes):
    if isinstance(outcome, StructuredFailure):
        status = f"FAILED ({outcome.error_type}): {outcome.message}"
    else:
        usage = outcome.metadata.usage
        status = (f"ok, {len(outcome.result.properties)} record(s), finish {outcome.metadata.finish_reason}, "
                  f"tokens {usage.prompt_tokens if usage else '?'} in / {usage.completion_tokens if usage else '?'} out")
    print(f"{request_label(request)}: {status}")
    if SHOW_REASONING and not isinstance(outcome, StructuredFailure):
        print("--- Reasoning ---\n" + (outcome.metadata.reasoning or "[No reasoning returned; is THINKING on?]") + "\n--- End ---")

print(f"\nRequests {result.requests}; records {len(result.records)} (linked to a sample {sum(r.linked for r in result.records)}); "
      f"failures {len(result.failures)}; "
      f"tokens {result.prompt_tokens:,} in / {result.completion_tokens:,} out; complete: {result.complete}")

targets [1764764, 1764765, 1764766, 1764767, 1764768]: ok, 6 record(s), finish stop, tokens 11485 in / 9893 out

Requests 1; records 6 (linked to a sample 5); failures 0; dropped as malformed 0; tokens 11,485 in / 9,893 out; complete: True


## Results per sample

Every record with all fields, grouped by sample in the paper's order. Under each record: the flags, and for `p2` how the value was linked. Values linked to no sample come last; they do not become `Property` records.

In [6]:
RECORD_FIELDS = _schemas.RECORD_FIELDS
names = {s.sample_id: f"{s.sample_id}: {s.name} | {s.identifier}" for s in paper.samples}


def print_record(n, item):
    record = item.record.model_dump(mode="json")
    print(f"\n[{n}]")
    print(json.dumps({k: record.get(k) for k in RECORD_FIELDS}, ensure_ascii=False, indent=2))
    line = f"    flags: {', '.join(item.flags) or '-'}"
    if isinstance(item.record, _schemas.PaperPropertyRecord):
        line += (f"\n    match_basis: {item.record.match_basis} | paper_label: {item.record.paper_label!r} | "
                 f"identifying_data: {item.record.identifying_data!r}")
    print(line)


def print_block(title, items):
    print(f"\n--- {title}: {len(items)} record(s) ---")
    if not items:
        print("(no records)")
    for n, item in enumerate(items, 1):
        print_record(n, item)


print("=" * 80)
print(f"{PROMPT.upper()} ({extractor.mode}) - {PAPER_UID} - {PROPERTY}")
print("=" * 80)
for sample in paper.samples:
    print_block(names[sample.sample_id], [r for r in result.records if r.sample_id == sample.sample_id])
if PROMPT == "p2":
    print_block("No sample (values linked to no listed sample)", [r for r in result.records if r.sample_id is None])
for failure in result.failures:
    print(f"\nFAILED request for {failure.sample_ids}: {failure.error_type}: {failure.message}")

P2 (per_paper) - ID000752848 - melting_temperature

--- 1764764: Homo PBS | Homo PBS: 1 record(s) ---

[1]
{
  "property": "melting_temperature",
  "method": "DSC",
  "subtype": "peak",
  "value_raw": "114.2 °C",
  "value": 114.2,
  "value_type": "exact",
  "unit": "°C",
  "conditions": [
    {
      "name": "heating_rate",
      "type": "exact",
      "value": 10,
      "unit": "°C/min",
      "description": null
    }
  ],
  "origin": "measured",
  "description": "Melting temperature of the polymer matrix, taken as the melting peak in a DSC heating scan, recorded at 10 °C/min using a Q-2000 DSC; no temperature range, atmosphere, specimen form, or thermal history stated."
}
    flags: -
    match_basis: same_entry | paper_label: 'Homo PBS' | identifying_data: 'ηinh 0.86, Mn 37500 g/mol, Mw 69000 g/mol, PDI 1.84, Tg -15.1 °C, Tc 72.9 °C, PC time 2 h 20 min'

--- 1764765: PCN01 | PCN01: 1 record(s) ---

[1]
{
  "property": "melting_temperature",
  "method": "DSC",
  "subtype": "peak",
 

## Property records

The records as they would be stored in the `properties` table: one per value of a sample, the same for `p2` and `p3`. `clean` is true when there are no flags.

In [7]:
def conditions_text(conditions):
    parts = []
    for c in conditions:
        text = f"{c.name}={c.value}" + (f" {c.unit}" if c.unit else "")
        parts.append(text + (f" ({c.type})" if c.type != "exact" else ""))
    return "; ".join(parts)


rows = []
for p in properties:
    rows.append({
        "sample": names[p.sample_id], "property": p.property, "method": p.method, "subtype": p.subtype,
        "value_raw": p.value_raw, "value": p.value, "value_type": p.value_type, "unit": p.unit,
        "conditions": conditions_text(p.conditions), "origin": p.origin, "description": p.description,
        "flags": ", ".join(p.flags), "clean": p.clean,
    })
properties_table = pd.DataFrame(rows)
print(f"{len(properties)} property record(s); clean {sum(p.clean for p in properties)}")
if not properties_table.empty:
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(properties_table)

clean_values = {}
for p in properties:
    if p.clean:
        clean_values.setdefault(p.sample_id, []).append(p.value_raw)
summary = pd.DataFrame([{"sample": names[s.sample_id], "clean values": "; ".join(clean_values.get(s.sample_id, []))}
                        for s in paper.samples])
print("\nClean values per sample:")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(summary)

unlinked = [r for r in result.records if not r.linked]
if unlinked:
    print(f"\n{len(unlinked)} value(s) linked to no sample (not stored):",
          "; ".join(f"{r.record.value_raw} ({getattr(r.record, 'paper_label', '')})" for r in unlinked))

5 property record(s); clean 5


,sample,property,method,subtype,value_raw,value,value_type,unit,conditions,origin,description,flags,clean
0,1764764: Homo PBS | Homo PBS,melting_temperature,DSC,peak,114.2 °C,114.2,exact,°C,heating_rate=10 °C/min,measured,"Melting temperature of the polymer matrix, taken as the melting peak in a DSC heating scan, recorded at 10 °C/min using a Q-2000 DSC; no temperature range, atmosphere, specimen form, or thermal history stated.",,True
1,1764765: PCN01 | PCN01,melting_temperature,DSC,peak,114.0 °C,114.0,exact,°C,heating_rate=10 °C/min,measured,"Melting temperature of the polymer matrix, taken as the melting peak in a DSC heating scan, recorded at 10 °C/min using a Q-2000 DSC; no temperature range, atmosphere, specimen form, or thermal history stated.",,True
2,1764766: PCN03 | PCN03,melting_temperature,DSC,peak,114.0 °C,114.0,exact,°C,heating_rate=10 °C/min,measured,"Melting temperature of the polymer matrix, taken as the melting peak in a DSC heating scan, recorded at 10 °C/min using a Q-2000 DSC; no temperature range, atmosphere, specimen form, or thermal history stated.",,True
3,1764767: PCN05 | PCN05,melting_temperature,DSC,peak,114.0 °C,114.0,exact,°C,heating_rate=10 °C/min,measured,"Melting temperature of the polymer matrix, taken as the melting peak in a DSC heating scan, recorded at 10 °C/min using a Q-2000 DSC; no temperature range, atmosphere, specimen form, or thermal history stated.",,True
4,1764768: PCN10 | PCN10,melting_temperature,DSC,peak,113.8 °C,113.8,exact,°C,heating_rate=10 °C/min,measured,"Melting temperature of the polymer matrix, taken as the melting peak in a DSC heating scan, recorded at 10 °C/min using a Q-2000 DSC; no temperature range, atmosphere, specimen form, or thermal history stated.",,True



Clean values per sample:


,sample,clean values
0,1764764: Homo PBS | Homo PBS,114.2 °C
1,1764765: PCN01 | PCN01,114.0 °C
2,1764766: PCN03 | PCN03,114.0 °C
3,1764767: PCN05 | PCN05,114.0 °C
4,1764768: PCN10 | PCN10,113.8 °C



1 value(s) linked to no sample (not stored): 114.0 to 114.5 °C (semi-crystalline PBS synthesized)


In [ ]:
result.to_properties()

## Raw responses (optional)

In [ ]:
SHOW_RAW = False
if SHOW_RAW:
    for request, outcome in zip(requests, outcomes):
        print(f"--- {request_label(request)} ---")
        print(outcome.raw_response if outcome.raw_response else "[no response]")

## Timing: prompt 2 vs prompt 3 on six papers

Runs each prompt on the papers below for the properties in `TIMING_PROPERTIES`, the way a runner would: all papers at
once, with one shared limit of `MAX_CONCURRENCY` requests in flight (as the sample extraction ran 8 papers in parallel).

- Prompt 2: one request per paper (per batch of `BATCH_SIZE` targets), so up to `MAX_CONCURRENCY` papers run in parallel.
- Prompt 3: one request per sample; the requests of all papers share the same `MAX_CONCURRENCY` slots.

Per paper: `seconds` is from the paper's first request starting to its last request finishing (queue waiting before
the first request is not counted); `request_seconds` is the sum of its request durations. `total` is the wall time of
the whole run. Uses the settings above (`MODEL`, `THINKING`, `MAX_TOKENS`, `MAX_CONCURRENCY`, `BATCH_SIZE`); run the
cells above first.

In [ ]:
import time

TIMING_UIDS = [
    "ID000415115",  # 6   Chemically modified lignin in PCL nanofibers
    "ID000573272",  # 5   Glycolysis of waste PET + PTMG copolymers
    "ID000588793",  # 13  PEB-b-PTMO multiblock copolymers, elasticity
    "ID000732480",  # 6   PLA with thionoester linkages
    "ID000752848",  # 5   Cellulose nanocrystal trans-crystallisation
    "ID001032337",  # 6   Biobased thermoplastic elastomers
]
TIMING_PROPERTIES = [PROPERTY]   # one or more property keys, e.g. ["glass_transition_temperature", "melting_temperature"]

timing_papers = {}
for _uid in TIMING_UIDS:
    _paper, _title, _ = load_paper(_uid)
    timing_papers[_uid] = _schemas.PaperContext.model_validate(_paper.model_dump())
    print(f"{_uid}: {len(_paper.samples)} samples, {len(_paper.markdown_full_text):,} characters - {_title[:80]}")


async def time_prompt(prompt):
    """Run one prompt on every timing paper and property, all in parallel under one request limit."""
    cls = _extractor.PaperPropertyExtractor if prompt == "p2" else _extractor.SamplePropertyExtractor
    extractors = {prop: cls(backend, prop) for prop in TIMING_PROPERTIES}
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)   # shared by every request of the run

    async def timed(extractor_, request):
        async with semaphore:
            start = time.perf_counter()
            outcome = await extractor_.aextract(request)
            return outcome, start, time.perf_counter()

    async def one(uid, prop):
        paper_ctx, extractor_ = timing_papers[uid], extractors[prop]
        requests_ = (extractor_.build_requests(paper_ctx, BATCH_SIZE) if prompt == "p2"
                     else extractor_.build_requests(paper_ctx))
        timed_outcomes = await asyncio.gather(*(timed(extractor_, r) for r in requests_))
        res = extractor_.collect(paper_ctx, requests_, [o for o, _, _ in timed_outcomes])
        seconds = max(e for _, _, e in timed_outcomes) - min(s for _, s, _ in timed_outcomes)
        busy = sum(e - s for _, s, e in timed_outcomes)
        stored = res.to_properties()
        print(f"{prompt} {uid} {prop}: {seconds:.1f} s, {res.requests} request(s), {len(stored)} record(s), "
              f"{len(res.failures)} failure(s)")
        return res, {
            "paper_uid": uid, "property": prop, "samples": len(paper_ctx.samples), "requests": res.requests,
            "seconds": round(seconds, 1), "request_seconds": round(busy, 1),
            "records": len(stored), "clean": sum(p.clean for p in stored),
            "unlinked": sum(not r.linked for r in res.records), "failures": len(res.failures),
            "prompt_tokens": res.prompt_tokens, "completion_tokens": res.completion_tokens,
        }

    jobs = [(uid, prop) for uid in TIMING_UIDS for prop in TIMING_PROPERTIES]
    total_start = time.perf_counter()
    outputs = await asyncio.gather(*(one(uid, prop) for uid, prop in jobs))
    total = time.perf_counter() - total_start

    results_ = {job: res for job, (res, _) in zip(jobs, outputs)}
    table = pd.DataFrame([row for _, row in outputs])
    print(f"\n{prompt.upper()}: total {total:.1f} s for {len(jobs)} paper-property pair(s) with at most "
          f"{MAX_CONCURRENCY} requests in flight; {table.requests.sum()} request(s), {table.records.sum()} record(s) "
          f"({table.clean.sum()} clean), {table.prompt_tokens.sum():,} prompt / {table.completion_tokens.sum():,} "
          f"completion tokens ({table.completion_tokens.sum() / total:.0f} completion tokens/s overall)")
    with pd.option_context("display.max_rows", None):
        display(table)
    return table, results_, total

In [ ]:
# Prompt 2 (per paper) on the timing papers.
timing_p2, timing_results_p2, timing_total_p2 = await time_prompt("p2")

In [ ]:
# Prompt 3 (per sample, with context) on the timing papers, then the comparison with prompt 2 if it has been run.
timing_p3, timing_results_p3, timing_total_p3 = await time_prompt("p3")

if "timing_p2" in globals():
    keys = ["paper_uid", "property", "samples"]
    cols = ["requests", "seconds", "request_seconds", "records", "clean", "unlinked", "failures", "prompt_tokens", "completion_tokens"]
    comparison_timing = timing_p2[keys + cols].merge(timing_p3[keys + cols], on=keys, suffixes=("_p2", "_p3"))
    comparison_timing["time_ratio_p3_p2"] = (comparison_timing.seconds_p3 / comparison_timing.seconds_p2).round(1)
    comparison_timing["token_ratio_p3_p2"] = (comparison_timing.prompt_tokens_p3
                                              / comparison_timing.prompt_tokens_p2.where(comparison_timing.prompt_tokens_p2 > 0)).round(1)
    print(f"\nTotal time: P2 {timing_total_p2:.1f} s, P3 {timing_total_p3:.1f} s "
          f"(P3/P2 = {timing_total_p3 / timing_total_p2:.1f}x)")
    print(f"Prompt tokens: P2 {timing_p2.prompt_tokens.sum():,}, P3 {timing_p3.prompt_tokens.sum():,}")
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(comparison_timing[keys + [c + s for c in ["seconds", "requests", "records", "clean", "prompt_tokens"]
                                         for s in ("_p2", "_p3")] + ["time_ratio_p3_p2", "token_ratio_p3_p2"]])